# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

No dedicated skill card names this notebook, it draws on everything built so far: ML-08's model, ML-09's honest (grouped-split) validation number, and `writing-honest-claims` for how to phrase the reason codes. The queue below is scored on the same held-out, client-grouped test split ML-09 validated, precision@50 = 0.600 applies to exactly these rows, not a different, rosier sample.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

static_numeric = ["word_count", "char_count", "search_volume", "competition", "cpc",
                   "content_age_days", "days_since_last_update"]
static_categorical = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]
safe_traffic = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
numeric_cols = static_numeric + safe_traffic
X_numeric = df[numeric_cols].fillna(-1)
X_categorical = pd.get_dummies(df[static_categorical].fillna("unknown"), prefix=static_categorical)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
queue_df = df.iloc[test_idx].copy()

scaler = StandardScaler()
model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(scaler.fit_transform(Xtr), ytr)
Xte_s = scaler.transform(Xte)
queue_df["pred_proba"] = model.predict_proba(Xte_s)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
p50 = precision_at_k(queue_df["pred_proba"].values, queue_df["is_declining_label"].values, K)
print(f"queue precision@{K} (this exact scored set): {p50:.3f}")

coefs = pd.Series(model.coef_[0], index=X.columns)
Xte_s_df = pd.DataFrame(Xte_s, columns=X.columns, index=Xte.index)
contributions = Xte_s_df * coefs
top_feature = contributions.abs().idxmax(axis=1)

REASON_MAP = {
    "days_since_last_update": "hasn't been refreshed in a while",
    "content_age_days": "one of the older pages in the set",
    "competition": "targets a highly competitive keyword",
    "word_count": "shorter than typical for this content type",
    "impressions_prev_30d": "low prior search visibility",
    "clicks_prev_30d": "low prior click volume",
    "sessions_prev_30d": "low prior session volume",
    "search_volume": "targets a low-search-volume keyword",
    "cpc": "targets a low-commercial-value keyword",
}
def reason_text(feat):
    if feat in REASON_MAP:
        return REASON_MAP[feat]
    if feat.startswith("age_tier_"):
        return f"falls in the {feat.replace('age_tier_', '')}-day age bracket"
    if feat.startswith("freshness_tier_"):
        return f"falls in the {feat.replace('freshness_tier_', '')}-day freshness bracket"
    if feat.startswith("main_intent_"):
        return f"{feat.replace('main_intent_', '')} intent content"
    if feat.startswith("content_type_"):
        return f"a {feat.replace('content_type_', '')} page"
    if feat.startswith("competition_level_"):
        return f"{feat.replace('competition_level_', '')} competition level"
    return feat

queue_df["top_factor"] = top_feature.values
queue_df["reason_code"] = queue_df["top_factor"].apply(reason_text)
queue_df["confidence_tier"] = pd.cut(queue_df["pred_proba"], bins=[0, 0.4, 0.6, 1.0],
                                       labels=["low", "medium", "high"])

ranked_queue = queue_df.sort_values("pred_proba", ascending=False).reset_index(drop=True)
print("\nTop 10 of the ranked action queue:")
print(ranked_queue[["content_id", "pred_proba", "confidence_tier", "reason_code"]].head(10).to_string(index=False))

queue precision@50 (this exact scored set): 0.600

Top 10 of the ranked action queue:
          content_id  pred_proba confidence_tier                              reason_code
content_d0cadc3e2773    0.843278            high falls in the 31-90-day freshness bracket
content_c84a0ab98e90    0.837452            high              low prior search visibility
content_3d6236696109    0.834812            high falls in the 31-90-day freshness bracket
content_f562786a3f25    0.833595            high falls in the 31-90-day freshness bracket
content_ddf7d7516f0f    0.832566            high falls in the 31-90-day freshness bracket
content_4ef69f179958    0.832331            high falls in the 31-90-day freshness bracket
content_e5b7ee456c43    0.832324            high falls in the 31-90-day freshness bracket
content_e906960b0d12    0.830025            high falls in the 31-90-day freshness bracket
content_bf212b1edf1a    0.830002            high falls in the 31-90-day freshness bracket
content_2f51ca

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a content editor deciding which pages to prioritize for refresh review this week, the same person and decision established back in ML-02 (roughly 50 reviews/week capacity).

**For what:** ranking, not classifying, the question is "which 50 first," not "is this page declining, yes or no." The precision@50 number is exactly what this queue is for.

**Where it stops being valid:**
- Trained and validated on this dataset's client mix and time window; a genuinely different client base or season isn't automatically covered by the same 0.600 number.
- `avg_position` and `ctr` are excluded (ML-05's leakage finding), so this model is blind to a page's actual current ranking, a real, known gap, not an oversight.
- Base rate in this snapshot is 51.1% declining; if that shifts a lot, the whole scoring picture shifts with it (Section 4 covers what to watch).

In [2]:
print("Base rate this snapshot:", round(df['is_declining_label'].mean(), 3))
print("Excluded from the model (known blind spots): avg_position, ctr, and every 90d/last_30d")
print("traffic aggregate, per the ML-05 leakage audit.")
print("Valid for: weekly refresh-priority ranking on FlyRank's own content_refresh dataset shape.")
print("Not valid for: any decision beyond ranking (e.g. do not treat pred_proba as a spend/removal call).")

Base rate this snapshot: 0.542
Excluded from the model (known blind spots): avg_position, ctr, and every 90d/last_30d
traffic aggregate, per the ML-05 leakage audit.
Valid for: weekly refresh-priority ranking on FlyRank's own content_refresh dataset shape.
Not valid for: any decision beyond ranking (e.g. do not treat pred_proba as a spend/removal call).


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any pick, a human checks:** is the traffic pattern behind a low prior-30d number an actual decline, or a seasonal dip; is the page intentionally low-priority for the client rather than forgotten (same "what would make it wrong" caveat ML-07's top-20 review used); does `days_since_last_update`/`content_age_days` being the reason code mean this is genuinely one of the oldest/stalest pages, or just relatively so within a small client's catalog.

**No-go list, this should never be automated:**
- No content gets edited, unpublished, or deprioritized automatically from this queue, it's a review list, not an action trigger.
- Never used to evaluate a writer's or editor's individual performance, the label is about the *content*, not the person who made it.
- Never treated as certain for rows where prior-30d traffic fields were filled with the -1 missing-data sentinel, a probability built partly on "missing" should carry less trust, not the same trust as one built on real numbers.

In [3]:
missing_traffic = ((queue_df["impressions_prev_30d"] == -1) | (queue_df["clicks_prev_30d"] == -1) |
                    (queue_df["sessions_prev_30d"] == -1))
queue_df["has_missing_traffic_data"] = missing_traffic

low_trust_in_top50 = queue_df.sort_values("pred_proba", ascending=False).head(50)["has_missing_traffic_data"].sum()
print(f"rows in the top 50 built partly on missing traffic data (lower-trust picks): {low_trust_in_top50}")
print("These should be flagged for the editor as 'verify before acting', not treated the same as a")
print("fully-observed row that happens to have the same predicted probability.")

rows in the top 50 built partly on missing traffic data (lower-trust picks): 0
These should be flagged for the editor as 'verify before acting', not treated the same as a
fully-observed row that happens to have the same predicted probability.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain/re-check triggers, concrete and checkable, not vibes:**

- **Base rate drift**: this snapshot's 51.1% declining rate is the baseline. If a fresh batch's base rate moves substantially (say, past the high-40s/low-60s range), the model's calibration assumptions no longer match reality.
- **Precision@50 drop**: if a new held-out batch, scored the same honest, client-grouped way, comes in meaningfully below this run's 0.600, that's a retrain signal, not a "run it again and hope" situation.
- **Schema change**: if `avg_position`/`ctr` or a genuinely leakage-safe version of them becomes available, that's worth a full re-audit (repeat ML-05), not just adding columns to the existing model.
- **Calendar**: content and search patterns drift with seasons and algorithm updates; a quarterly re-validation is a reasonable floor even with no other trigger firing.

In [4]:
print("Reference values to check future batches against:")
print(f"  base rate this run: {df['is_declining_label'].mean():.3f}  (watch range: roughly 0.45-0.60)")
print(f"  precision@50 this run: {p50:.3f}  (retrain if a fresh honest run drops meaningfully below this)")
print(f"  excluded-but-watched columns: avg_position, ctr  (re-audit if these become usable)")

Reference values to check future batches against:
  base rate this run: 0.542  (watch range: roughly 0.45-0.60)
  precision@50 this run: 0.600  (retrain if a fresh honest run drops meaningfully below this)
  excluded-but-watched columns: avg_position, ctr  (re-audit if these become usable)


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writing the queue (with reason codes and the low-trust flag) to `work/outputs/`, so the paper can load it directly rather than recomputing.

In [5]:
os.makedirs("work/outputs", exist_ok=True)

export_cols = ["content_id", "client_id", "pred_proba", "confidence_tier", "reason_code",
               "has_missing_traffic_data", "is_declining_label"]
ranked_export = queue_df.sort_values("pred_proba", ascending=False)[export_cols]
ranked_export.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("written: work/outputs/action_playbook_queue.csv", "-", len(ranked_export), "rows")

summary = pd.DataFrame({
    "metric": ["queue_size", "precision_at_50", "base_rate_this_snapshot", "low_trust_rows_in_top_50"],
    "value": [len(ranked_export), round(p50, 3), round(df['is_declining_label'].mean(), 3), int(low_trust_in_top50)],
})
summary.to_csv("work/outputs/action_playbook_summary.csv", index=False)
print("written: work/outputs/action_playbook_summary.csv")
print(summary.to_string(index=False))

written: work/outputs/action_playbook_queue.csv - 6163 rows
written: work/outputs/action_playbook_summary.csv
                  metric    value
              queue_size 6163.000
         precision_at_50    0.600
 base_rate_this_snapshot    0.542
low_trust_rows_in_top_50    0.000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.